# 感情AI: 内部信号と感情語の接続実験 — C0(対照条件)の実行(Colab / T4)

事前登録 `事前登録_内部信号と感情語の接続実験.md` の4章「C0(対照): 学習なし。同一プロンプト・環境で推論のみ」を、
追記欄「2026-09-12」で確定した標本(**15 seed × 30 episode/seed**)と環境定数(B0=340、閾値85、
U_OPT/MIN/MAX=0.7/0.0/2.5、課題は add/sub/mul/count、温度1.0)で実行する。

このノートブックは**学習を一切行わず、分析も行わない**。`run_c0.py` を動かして、seed ごとの全 StepRecord を
Google Drive(`MyDrive/EmotionalAI/c0_v2/c0_seedNN.json`)に保存するだけ。
粒度スコアの算出(`analyze_granularity.py`)は、15 seed がすべて完了してから別の手順として行う。

**v2(2026-09-13)**: C0(v1)では感情語がほぼ出なかった(使用率0.19%)ため、追記欄「2026-09-13(2)」に従い
プロンプトに相手の枠を加え(status: ラベルつき)、辞書に定型句の除外規則を入れた。v1 のデータは `data/c0_v1/` に残し、
v2 の保存先は Drive の `MyDrive/EmotionalAI/c0_v2/`(試走は `c0_v2_trial/`)。**先に「5b. 試走」で使用率を確認し、
20% 未満なら分岐D(停止して報告)、20% 以上なら指示を受けてから「6. 実行」に進む。**

- 所要時間の目安: 速度実測(12.47秒/episode)から 450 episode ≈ 94分 + モデル読み込み(v1 実績: 約7分/seed)。
- **途中で切れたら**: 「6. 実行」のセルをもう一度実行する。完了済みの seed は自動的に飛ばし、残りだけ実行する
  (seed 単位で保存しているので、失うのは最大で実行中だった 1 seed 分)。
- 実行順序: 1) GPU確認 → 2) GitHubからclone → 3) 依存インストール → 4) Google Driveマウント → 5) 動作確認 → 6) 実行 → 7) 保存状況の確認


## 1. GPU確認

ランタイムのタイプが「T4 GPU」になっていることを確認する。

In [ ]:
!nvidia-smi
import torch
print("CUDA利用可能:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPUが見つからない。上部メニューの「ランタイム」→「ランタイムのタイプを変更」で"
        "ハードウェアアクセラレータを「T4 GPU」に設定してから、このセルからやり直すこと。"
    )


## 2. GitHubからclone

`seina369/homeostatic-agent-experiments`(公開リポジトリ)。既にあれば `git pull` で最新にする。

In [ ]:
import os

REPO_URL = "https://github.com/seina369/homeostatic-agent-experiments.git"
REPO_DIR = "/content/homeostatic-agent-experiments"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
LLM_DIR = os.path.join(REPO_DIR, "llm_grounding")
!ls {LLM_DIR}/run_c0.py {LLM_DIR}/seed_records.py {LLM_DIR}/torch_qwen_policy.py {LLM_DIR}/emotion_grounding_env.py
!cd {REPO_DIR} && git log --oneline -1


## 3. 依存のインストール

`llm_grounding/requirements.txt`(torch は Colab 既存の CUDA 版をそのまま使う)。

In [ ]:
!pip install -q -r {LLM_DIR}/requirements.txt


## 4. Google Driveをマウント

保存先 `MyDrive/EmotionalAI/c0_v2/`(試走用に `c0_v2_trial/`)を用意する。アクセス許可のダイアログが出るので、許可する(タイムアウトしたらこのセルだけ再実行)。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

C0_DIR = "/content/drive/MyDrive/EmotionalAI/c0_v2"        # v1 の c0/ は残したまま触らない
TRIAL_DIR = "/content/drive/MyDrive/EmotionalAI/c0_v2_trial"
os.makedirs(C0_DIR, exist_ok=True)
os.makedirs(TRIAL_DIR, exist_ok=True)
print("保存先(本番):", C0_DIR, "既存:", sorted(f for f in os.listdir(C0_DIR) if f.endswith(".json")))
print("保存先(試走):", TRIAL_DIR, "既存:", sorted(f for f in os.listdir(TRIAL_DIR) if f.endswith(".json")))


## 5. 動作確認

本番前に、環境定数が事前登録の確定値になっていること、モデルが読み込めて `respond()` が1回動くことを確認する。

In [ ]:
import sys
sys.path.insert(0, LLM_DIR)
import emotion_grounding_env as E
from seed_records import env_constants
from torch_qwen_policy import TorchPolicy

print("環境定数:", env_constants())
assert (E.B0, E.BUDGET_LOW_THRESHOLD, E.U_OPT, E.U_MIN, E.U_MAX) == (340, 85.0, 0.7, 0.0, 2.5), "事前登録の確定値と違う"
assert E.PROMPT_VERSION == 2, "v2 のプロンプトになっていない(git pull を確認)"
print(E.PROMPT_TEMPLATE)

_p = TorchPolicy(verbose=False)
_r = _p.respond(E.PROMPT_TEMPLATE.format(task="What is 23 + 19?", budget=340, error=0, uncertainty=0.70))
print(repr(_r.text[:120]), f"n_tokens={_r.n_tokens} mean_entropy={_r.mean_entropy:.3f}")
del _p, _r
torch.cuda.empty_cache()


## 5b. 試走(3 seed × 10 episode)— 分岐Dの判定

修正後のプロンプト・辞書で感情語の使用率を確認する。**20% 未満なら分岐D**(この系では従属変数が測れないと判断して停止・報告)。20% 以上なら、指示を受けてから「6. 実行」へ。試走のデータは本番の標本に含めない。

In [ ]:
import time
t0 = time.time()
!python3 {LLM_DIR}/run_c0.py --policy torch --seeds 3 --episodes 10 --temperature 1.0 --out-dir "{TRIAL_DIR}"
print(f"所要時間: {(time.time() - t0) / 60:.1f}分")
print()
# 使用率と副指標(分析コードは本番と同じものを、変更せずに使う)
!python3 {LLM_DIR}/analyze_granularity.py --dir "{TRIAL_DIR}" --out "{TRIAL_DIR}/trial_analysis.json"


## 6. 実行(15 seed × 30 episode)

途中で切れたら、このセルをもう一度実行する(完了済み seed は飛ばされる)。

In [ ]:
import time
t0 = time.time()
!python3 {LLM_DIR}/run_c0.py --policy torch --seeds 15 --episodes 30 --temperature 1.0 --out-dir "{C0_DIR}"
print(f"所要時間: {(time.time() - t0) / 60:.1f}分")


## 7. 保存状況の確認

seed ごとのファイルが揃っているかを見るだけ(中身の分析はしない)。

In [ ]:
import json, glob
files = sorted(glob.glob(f"{C0_DIR}/c0_seed*.json"))
print(f"{len(files)} / 15 seed 保存済み(v2: {C0_DIR})")
for f in files:
    with open(f, encoding="utf-8") as fh:
        p = json.load(fh)
    print(f"  {os.path.basename(f)}: complete={p.get('complete')} episodes={p.get('n_episodes')} "
          f"steps={p.get('n_steps')} elapsed={p.get('elapsed_seconds', 0) / 60:.1f}min model={p.get('policy', {}).get('model')}")


## このあと

15 seed がすべて `complete=True` になったら、C0 の実行は終わり。粒度スコアと副指標の算出は
`analyze_granularity.py --dir <c0フォルダ>` で行う(事前登録5章)。分析は C0 完了後に、
事前登録の手順どおり別に行い、結果と判断は追記欄に日付つきで記録する。
